# EXPERIMENT 1 — SPATIAL UPSAMPLING QUALITY
## AnyUp3D Results · Section 4.2.3
**Linear probe mIoU on ADE20K · 3 conditions × 3 seeds**

Conditions: `Bilinear` | `2D AnyUp` | `AnyUp3D`

## Cell 1 — Mount Drive + Install

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# HuggingFace datasets for ADE20K (scene_parse_150); scipy for t-test
# These are small and fast to install
get_ipython().system('pip install datasets scipy -q')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR} — skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Repo already present at /content/anyup — skipping clone.
Working directory: /content/anyup


## Cell 2 — Imports + Config

In [4]:
import os
import sys
import json
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from datasets import load_dataset
from scipy.stats import ttest_rel
from pathlib import Path

from anyup.model import AnyUp                # AnyUp3D (the 3D model class)

# ── Config — single source of truth ──────────────────────────────────────────

CKPT_ANYUP3D   = "/content/drive/MyDrive/anyup3d/checkpoints/run_01/anyup3d_step10000.pth"
OUTPUT_DIR     = "/content/drive/MyDrive/anyup3d/results/exp1"

SEEDS          = [42, 123, 456]     # 3 seeds for paired t-test; do not change order

# ADE20K (scene_parse_150): 150 classes, annotation values 1-150, 0 = unlabeled/ignore
NUM_CLASSES    = 150
IGNORE_INDEX   = 0                  # annotation value 0 = unlabeled; excluded from mIoU

IMG_SIZE       = 224                # ↓ reduce to 112 to halve memory; also update FEAT_SIZE
FEAT_SIZE      = 7                  # Video Swin-T norm-layer spatial output (224/32); depends on IMG_SIZE
FEAT_DIM       = 768                # Video Swin-T final feature dim; change if backbone changes

T_BACKBONE     = 2                  # ↓ min T for swin3d_t (temporal patch stride=2); frame repeated twice
                                    #   if you get temporal errors, increase to 4

BATCH_SIZE     = 8                  # ↓ reduce to 4 if OOM on T4; increase to 16 if memory allows
                                    #   affects: train loop, eval loop, figure generation

NUM_EPOCHS     = 20                 # ↓ reduce to 10 for a faster run; 20 is good for a clean result
LR             = 1e-3               # head-only learning rate; backbone + AnyUp3D are frozen
LR_DECAY_STEP  = 10                 # StepLR step size in epochs; depends on NUM_EPOCHS

# AnyUp3D constructor args — must match the checkpoint
ANYUP3D_CFG = dict(
    input_dim    = 3,
    qk_dim       = 128,
    kernel_size  = 1,
    window_ratio = 0.1,
    num_heads    = 4,
    t_k          = 1,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device : {DEVICE}")
print(f"IMG_SIZE={IMG_SIZE}, FEAT_SIZE={FEAT_SIZE}, FEAT_DIM={FEAT_DIM}, BATCH_SIZE={BATCH_SIZE}")

Device : cuda
IMG_SIZE=224, FEAT_SIZE=7, FEAT_DIM=768, BATCH_SIZE=8


## Cell 3 — Dataset (ADE20K via HuggingFace)

In [5]:
# ── Cell 3 — Dataset (ADE20K direct download from MIT) ───────────────────────
import os, glob
from pathlib import Path

ADE_DIR = Path("/content/ADEChallengeData2016")
FEAT_CACHE_DIR = Path("/content/drive/MyDrive/anyup3d/feat_cache")
FEAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ADE = Path("/content/drive/MyDrive/anyup3d/ADEChallengeData2016.zip")

if not ADE_DIR.exists():
    if DRIVE_ADE.exists():
        print("Copying ADE20K from Drive...")
        os.system(f"cp {DRIVE_ADE} /content/ade20k.zip")
    else:
        print("Downloading ADE20K from MIT (~900 MB)...")
        os.system("wget -q http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip -O /content/ade20k.zip")
        os.system(f"cp /content/ade20k.zip {DRIVE_ADE}")
    os.system("unzip -q /content/ade20k.zip -d /content/")

print(f"ADE20K ready at {ADE_DIR}  ✓")

# ── Transforms ────────────────────────────────────────────────────────────────
_img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_denorm_mean = torch.tensor([0.485, 0.456, 0.406])
_denorm_std  = torch.tensor([0.229, 0.224, 0.225])

def denorm(t):
    return (t * _denorm_std[:, None, None] + _denorm_mean[:, None, None]).clamp(0, 1)

# ── Raw image dataset (used only for precomputation in Cell 4) ────────────────
class ADE20KDataset(Dataset):
    def __init__(self, split):
        folder = "training" if split == "train" else "validation"
        self.imgs  = sorted((ADE_DIR / "images"      / folder).glob("*.jpg"))
        self.masks = sorted((ADE_DIR / "annotations" / folder).glob("*.png"))
        assert len(self.imgs) == len(self.masks) and len(self.imgs) > 0

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img  = _img_tf(Image.open(self.imgs[idx]).convert("RGB"))
        mask = Image.open(self.masks[idx])
        mask = transforms.functional.resize(
            mask, (IMG_SIZE, IMG_SIZE),
            interpolation=transforms.InterpolationMode.NEAREST,
        )
        mask = torch.from_numpy(np.array(mask)).long()
        return img, mask

# ── Cached dataset (used for all training — no backbone forward needed) ────────
class CachedFeatDataset(Dataset):
    """Loads precomputed (img, feat, mask) triples saved by Cell 4."""
    def __init__(self, split_name):
        path = FEAT_CACHE_DIR / f"{split_name}_cache.pt"
        assert path.exists(), f"Cache not found at {path} — run Cell 4 first."
        data = torch.load(path, map_location="cpu")
        self.feats = data["feats"].float()   # (N, FEAT_DIM, FEAT_SIZE, FEAT_SIZE)
        self.imgs  = data["imgs"].float()    # (N, 3, IMG_SIZE, IMG_SIZE)
        self.masks = data["masks"].long()    # (N, IMG_SIZE, IMG_SIZE)

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        return self.imgs[idx], self.feats[idx], self.masks[idx]

# ── Instantiate raw datasets (for precomputation only) ────────────────────────
_train_ds = ADE20KDataset("train")
_val_ds   = ADE20KDataset("validation")
print(f"Raw — Train: {len(_train_ds)}  Val: {len(_val_ds)}")

# CachedFeatDataset is instantiated at the bottom of Cell 4,
# after precomputation has run and the cache files exist.

ADE20K ready at /content/ADEChallengeData2016  ✓
Raw — Train: 20210  Val: 2000


## Cell 4 — Backbone (Video Swin-T) + Shape Verification

In [ ]:
# ── Cell 4 — Backbone + Precompute Features ───────────────────────────────────
_backbone_feat = {}

def _hook_fn(module, inp, out):
    _backbone_feat['norm'] = out.detach().permute(0, 4, 1, 2, 3).contiguous()

_backbone = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1).to(DEVICE)
_backbone.eval()
for p in _backbone.parameters():
    p.requires_grad_(False)
_hook_handle = _backbone.norm.register_forward_hook(_hook_fn)

def extract_backbone_features(imgs_4d):
    x = imgs_4d.unsqueeze(2).expand(-1, -1, T_BACKBONE, -1, -1).contiguous()
    with torch.no_grad():
        _backbone(x)
    return _backbone_feat['norm']   # (B, FEAT_DIM, 1, FEAT_SIZE, FEAT_SIZE)

# Shape verification
with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _feat  = extract_backbone_features(_dummy)
assert _feat.shape == (1, FEAT_DIM, 1, FEAT_SIZE, FEAT_SIZE), \
    f"Shape mismatch: {tuple(_feat.shape)}"
print(f"Backbone output shape: {tuple(_feat.shape)}  ✓")

# ── Precompute features for both splits ───────────────────────────────────────
# Runs backbone once over all images and saves (feat, img, mask) to Drive.
# Skipped automatically on rerun if files already exist.

FEAT_CACHE_DIR = Path("/content/drive/MyDrive/anyup3d/feat_cache")
FEAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def precompute_split(dataset, split_name):
    out_path = FEAT_CACHE_DIR / f"{split_name}_cache.pt"
    if out_path.exists():
        print(f"Cache exists for {split_name}, skipping.")
        return

    print(f"Precomputing {split_name} features ({len(dataset)} images)...")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

    CHUNK_SIZE = 500   # ↓ reduce to 250 if still OOM; each chunk ~150 MB
    chunk_feats, chunk_imgs, chunk_masks = [], [], []
    chunk_paths = []
    chunk_idx   = 0
    total_seen  = 0

    for imgs, masks in loader:
        imgs  = imgs.to(DEVICE)
        feats = extract_backbone_features(imgs).squeeze(2).cpu().to(torch.float16)
        chunk_feats.append(feats)
        chunk_imgs.append(imgs.cpu().to(torch.float16))
        chunk_masks.append(masks.to(torch.int16))
        total_seen += imgs.size(0)

        if total_seen % CHUNK_SIZE < BATCH_SIZE:
            # Save this chunk to disk and immediately free RAM
            chunk_path = FEAT_CACHE_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
            torch.save({
                "feats" : torch.cat(chunk_feats,  dim=0),
                "imgs"  : torch.cat(chunk_imgs,   dim=0),
                "masks" : torch.cat(chunk_masks,  dim=0),
            }, chunk_path)
            chunk_paths.append(chunk_path)
            chunk_feats, chunk_imgs, chunk_masks = [], [], []
            torch.cuda.empty_cache()
            chunk_idx += 1
            print(f"  {total_seen}/{len(dataset)}  (chunk {chunk_idx} saved)")

    # Save any remaining samples
    if chunk_feats:
        chunk_path = FEAT_CACHE_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
        torch.save({
            "feats" : torch.cat(chunk_feats,  dim=0),
            "imgs"  : torch.cat(chunk_imgs,   dim=0),
            "masks" : torch.cat(chunk_masks,  dim=0),
        }, chunk_path)
        chunk_paths.append(chunk_path)
        print(f"  {total_seen}/{len(dataset)}  (final chunk saved)")

    # Merge all chunks into one file
    print("Merging chunks...")
    all_feats = torch.cat([torch.load(p)["feats"] for p in chunk_paths], dim=0)
    all_imgs  = torch.cat([torch.load(p)["imgs"]  for p in chunk_paths], dim=0)
    all_masks = torch.cat([torch.load(p)["masks"] for p in chunk_paths], dim=0)
    torch.save({"feats": all_feats, "imgs": all_imgs, "masks": all_masks}, out_path)

    # Clean up chunk files
    for p in chunk_paths:
        p.unlink()

    print(f"Saved → {out_path}  ({len(all_feats)} samples)")

precompute_split(_train_ds, "train")
precompute_split(_val_ds,   "val")
print("Precomputation done  ✓")

Backbone output shape: (1, 768, 1, 7, 7)  ✓
Precomputing train features (20210 images)...
  504/20210  (chunk 1 saved)
  1000/20210  (chunk 2 saved)
  1504/20210  (chunk 3 saved)
  2000/20210  (chunk 4 saved)
  2504/20210  (chunk 5 saved)
  3000/20210  (chunk 6 saved)
  3504/20210  (chunk 7 saved)
  4000/20210  (chunk 8 saved)
  4504/20210  (chunk 9 saved)
  5000/20210  (chunk 10 saved)
  5504/20210  (chunk 11 saved)
  6000/20210  (chunk 12 saved)
  6504/20210  (chunk 13 saved)
  7000/20210  (chunk 14 saved)
  7504/20210  (chunk 15 saved)
  8000/20210  (chunk 16 saved)
  8504/20210  (chunk 17 saved)
  9000/20210  (chunk 18 saved)
  9504/20210  (chunk 19 saved)
  10000/20210  (chunk 20 saved)
  10504/20210  (chunk 21 saved)
  11000/20210  (chunk 22 saved)
  11504/20210  (chunk 23 saved)
  12000/20210  (chunk 24 saved)
  12504/20210  (chunk 25 saved)
  13000/20210  (chunk 26 saved)
  13504/20210  (chunk 27 saved)
  14000/20210  (chunk 28 saved)
  14504/20210  (chunk 29 saved)
  15000/202

In [ ]:
_train_cached = CachedFeatDataset("train")
_val_cached   = CachedFeatDataset("val")
print(f"Cached train: {len(_train_cached)}  val: {len(_val_cached)}")

## Cell 5 — Models: AnyUp3D, 2D AnyUp, Bilinear

In [ ]:
# ── Cell 5 — Models: AnyUp3D, 2D AnyUp, Bilinear ─────────────────────────────

class TStage:
    def __init__(self, t=None, start_step=None, batch_size=None):
        self.t          = t
        self.start_step = start_step
        self.batch_size = batch_size

# ── AnyUp3D ───────────────────────────────────────────────────────────────────
_anyup3d = AnyUp(
    input_dim       = 3,
    qk_dim          = 128,
    kernel_size     = 1,
    kernel_size_lfu = 5,
    window_ratio    = 0.1,
    num_heads       = 4,
    t_k             = 1,
).to(DEVICE).eval()

ckpt = torch.load(CKPT_ANYUP3D, map_location="cpu", weights_only=False)

sd = ckpt["anyup"] if "anyup" in ckpt else ckpt.get(
    "model_state_dict", ckpt.get("model", ckpt.get("state_dict", ckpt))
)
for k in ("step", "global_step", "t_stage"):
    if k in ckpt:
        print(f"  checkpoint {k}: {ckpt[k]}")

sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v
      for k, v in sd.items()}

missing, unexpected = _anyup3d.load_state_dict(sd, strict=False)
print(f"AnyUp3D — {len(missing)} missing  {len(unexpected)} unexpected")
if missing:    print(f"  missing (first 5):    {missing[:5]}")
if unexpected: print(f"  unexpected (first 5): {unexpected[:5]}")
if not missing and not unexpected:
    print("Strict match — all keys loaded cleanly  ✓")

for p in _anyup3d.parameters():
    p.requires_grad_(False)

# ── 2D AnyUp ──────────────────────────────────────────────────────────────────
# Remove /content/anyup from sys.path so hub loads the original 2D class,
# not our 3D AnyUp which has the same module name.
from anyup.model_2d import AnyUp2D
_anyup2d = AnyUp2D().to(DEVICE)
_ckpt2d  = torch.hub.load_state_dict_from_url(
    "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth",
    map_location=DEVICE
)
_anyup2d.load_state_dict(_ckpt2d)
_anyup2d.eval()
for p in _anyup2d.parameters():
    p.requires_grad_(False)
print("2D AnyUp loaded  ✓")

# ── Upsample functions ────────────────────────────────────────────────────────
@torch.no_grad()
def upsample_bilinear(imgs, feats_5d):
    feat_4d = feats_5d.squeeze(2)
    return F.interpolate(feat_4d, size=(IMG_SIZE, IMG_SIZE),
                         mode='bilinear', align_corners=False)

@torch.no_grad()
def upsample_anyup2d(imgs, feats_5d):
    feat_4d   = feats_5d.squeeze(2)
    feat_norm = F.normalize(feat_4d, dim=1)
    return _anyup2d(imgs, feat_norm, output_size=(IMG_SIZE, IMG_SIZE))

@torch.no_grad()
def upsample_anyup3d(imgs, feats_5d):
    imgs_5d   = imgs.unsqueeze(2)
    feat_norm = F.normalize(feats_5d, dim=1)
    out_5d    = _anyup3d(imgs_5d, feat_norm, output_size=(IMG_SIZE, IMG_SIZE))
    return out_5d.squeeze(2)

UPSAMPLERS = {
    "bilinear" : upsample_bilinear,
    "anyup2d"  : upsample_anyup2d,
    "anyup3d"  : upsample_anyup3d,
}

## Cell 6 — Linear Head + mIoU Utilities

In [ ]:
def make_head():
    """Fresh 1×1 conv linear segmentation head. One per seed."""
    return nn.Conv2d(FEAT_DIM, NUM_CLASSES, kernel_size=1).to(DEVICE)  # FEAT_DIM=768; depends on backbone

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def compute_miou(conf_mat):
    """
    conf_mat : (NUM_CLASSES, NUM_CLASSES) int64 tensor — rows=GT, cols=pred
    Returns  : float mIoU in [0, 100]
    """
    tp    = conf_mat.diag().float()
    fp    = (conf_mat.sum(0) - conf_mat.diag()).float()
    fn    = (conf_mat.sum(1) - conf_mat.diag()).float()
    iou   = tp / (tp + fp + fn + 1e-10)
    valid = conf_mat.sum(1) > 0          # only classes present in GT
    return iou[valid].mean().item() * 100.0

def update_conf_mat(conf_mat, logits, targets):
    """
    logits  : (B, NUM_CLASSES, H, W)   — raw head output
    targets : (B, H, W)                — annotation values 1–150, 0=ignore
    conf_mat is updated in-place.
    """
    preds = logits.argmax(dim=1)        # (B, H, W) — class indices 0–149
    mask  = (targets != IGNORE_INDEX)   # ignore unlabeled pixels

    tgt_flat  = targets[mask].cpu()
    pred_flat = preds[mask].cpu()

    # Annotation values 1–150 → head indices 0–149
    tgt_flat = tgt_flat - 1
    valid = (tgt_flat >= 0) & (tgt_flat < NUM_CLASSES) & \
            (pred_flat >= 0) & (pred_flat < NUM_CLASSES)
    tgt_flat  = tgt_flat[valid]
    pred_flat = pred_flat[valid]

    idx = tgt_flat * NUM_CLASSES + pred_flat
    conf_mat += torch.bincount(
        idx, minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(NUM_CLASSES, NUM_CLASSES)

## Cell 7 — Train + Evaluate Functions

In [ ]:
# ── Cell 7 — Train + Evaluate (cached features) ───────────────────────────────

def train_one_epoch(head, optimizer, upsample_fn, loader):
    head.train()
    total_loss = 0.0
    n_batches  = 0
    for imgs, feats, masks in loader:
        imgs  = imgs.to(DEVICE)                  # (B, 3, H, W)
        feats = feats.unsqueeze(2).to(DEVICE)    # (B, FEAT_DIM, 1, FEAT_SIZE, FEAT_SIZE)
        masks = masks.to(DEVICE) - 1             # shift 1–150 → 0–149; 0 → -1 (ignore)

        up     = upsample_fn(imgs, feats)        # (B, FEAT_DIM, H, W)
        logits = head(up)                        # (B, NUM_CLASSES, H, W)
        loss   = F.cross_entropy(logits, masks, ignore_index=-1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / max(n_batches, 1)

@torch.no_grad()
def evaluate_miou(head, upsample_fn, loader):
    head.eval()
    conf_mat = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    for imgs, feats, masks in loader:
        imgs  = imgs.to(DEVICE)
        feats = feats.unsqueeze(2).to(DEVICE)
        masks = masks.to(DEVICE)

        up     = upsample_fn(imgs, feats)
        logits = head(up)
        update_conf_mat(conf_mat, logits, masks)

    return compute_miou(conf_mat)

def run_seed(seed, condition_name):
    print(f"  seed={seed}  condition={condition_name}")
    set_seed(seed)

    upsample_fn = UPSAMPLERS[condition_name]
    head        = make_head()
    optimizer   = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    scheduler   = torch.optim.lr_scheduler.StepLR(optimizer,
                      step_size=LR_DECAY_STEP, gamma=0.1)

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(_train_cached, batch_size=BATCH_SIZE,  # ↓ BATCH_SIZE
                              shuffle=True, num_workers=2,
                              pin_memory=True, generator=g)
    val_loader   = DataLoader(_val_cached,   batch_size=BATCH_SIZE,  # ↓ BATCH_SIZE
                              shuffle=False, num_workers=2,
                              pin_memory=True)

    for epoch in range(1, NUM_EPOCHS + 1):
        loss = train_one_epoch(head, optimizer, upsample_fn, train_loader)
        scheduler.step()
        if epoch % 5 == 0 or epoch == NUM_EPOCHS:
            miou = evaluate_miou(head, upsample_fn, val_loader)
            print(f"    epoch {epoch:02d}/{NUM_EPOCHS}  loss={loss:.4f}  mIoU={miou:.2f}")

    return evaluate_miou(head, upsample_fn, val_loader), head

## Cell 8 — Main Experiment (3 conditions × 3 seeds)
This is the longest-running cell. On a T4:
  - Each epoch ≈ 25–40 s (backbone forward frozen, head backward trivial)
  - 20 epochs × 3 conditions × 3 seeds ≈ 4–5 hours total
  - Results are saved to Drive after each condition in case of runtime disconnect.

In [ ]:
CONDITIONS = ["bilinear", "anyup2d", "anyup3d"]

# results[condition] = [miou_seed0, miou_seed1, miou_seed2]
results = {c: [] for c in CONDITIONS}

# heads_last[condition] = head trained on last seed (seed[2]) — used for qualitative figure
heads_last = {}

for condition in CONDITIONS:
    print(f"\n{'─'*60}")
    print(f"Condition: {condition.upper()}")
    print(f"{'─'*60}")
    for seed in SEEDS:                               # 3 seeds
        miou, head = run_seed(seed, condition)
        results[condition].append(miou)
        print(f"  → final mIoU: {miou:.2f}")

    heads_last[condition] = head

    # Save partial results to Drive after each condition (crash safety)
    _save_path = os.path.join(OUTPUT_DIR, "exp1_results_partial.json")
    with open(_save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved partial results → {_save_path}")

# Final save
_final_path = os.path.join(OUTPUT_DIR, "exp1_results_final.json")
with open(_final_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nFinal results saved → {_final_path}")

## Cell 9 — Results Table + Paired t-test

In [ ]:
print("\n" + "═"*62)
print(f"{'EXPERIMENT 1 — SPATIAL UPSAMPLING QUALITY':^62}")
print(f"{'ADE20K Linear Probe · mIoU (%) · 3 seeds':^62}")
print("═"*62)
print(f"{'Condition':<18} {'Seed 1':>8} {'Seed 2':>8} {'Seed 3':>8}  {'Mean':>8}  {'Std':>6}")
print("─"*62)

means = {}
stds  = {}
for condition in CONDITIONS:
    scores = results[condition]
    mu  = float(np.mean(scores))
    std = float(np.std(scores, ddof=1))
    means[condition] = mu
    stds[condition]  = std
    label = {"bilinear": "Bilinear",
             "anyup2d" : "2D AnyUp (per-frame)",
             "anyup3d" : "AnyUp3D (ours)"}[condition]
    print(f"{label:<18} "
          f"{scores[0]:>8.2f} {scores[1]:>8.2f} {scores[2]:>8.2f}  "
          f"{mu:>8.2f}  {std:>6.2f}")

print("═"*62)

# ── Paired t-test: bilinear vs AnyUp3D ────────────────────────────────────────
t_stat, p_val = ttest_rel(results["anyup3d"], results["bilinear"])
print(f"\nPaired t-test  AnyUp3D vs Bilinear:")
print(f"  t = {t_stat:.4f}   p = {p_val:.4f}   {'p < 0.05  ✓' if p_val < 0.05 else 'NOT significant'}")

# ── Paired t-test: anyup3d vs anyup2d ─────────────────────────────────────────
t2, p2 = ttest_rel(results["anyup3d"], results["anyup2d"])
print(f"\nPaired t-test  AnyUp3D vs 2D AnyUp:")
print(f"  t = {t2:.4f}   p = {p2:.4f}   {'p < 0.05  ✓' if p2 < 0.05 else 'NOT significant'}")

# Save summary
summary = {
    "means"                 : means,
    "stds"                  : stds,
    "raw"                   : results,
    "t_anyup3d_vs_bilinear" : {"t": t_stat, "p": p_val},
    "t_anyup3d_vs_anyup2d"  : {"t": t2, "p": p2},
}
with open(os.path.join(OUTPUT_DIR, "exp1_summary.json"), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved → {OUTPUT_DIR}/exp1_summary.json")

## Cell 10 — Qualitative Figure
4 rows: one per example image. 5 columns: Original | GT | Bilinear | 2D AnyUp | AnyUp3D

In [ ]:
# ── Colour palette (150 colours, seeded for reproducibility) ──────────────────
np.random.seed(0)
_palette = np.random.randint(0, 255, size=(NUM_CLASSES + 1, 3), dtype=np.uint8)
_palette[0] = [0, 0, 0]            # class 0 (ignore) = black

def mask_to_rgb(mask_np):
    """mask_np: (H, W) int64 with values 0–150 → (H, W, 3) RGB"""
    return _palette[mask_np]

def predict_mask(head, upsample_fn, imgs, feats_5d):
    """Returns (B, H, W) predicted class indices 0–149 (head space)."""
    with torch.no_grad():
        up     = upsample_fn(imgs, feats_5d)
        logits = head(up)
        return logits.argmax(dim=1).cpu().numpy()

# ── Sample 4 validation images ─────────────────────────────────────────────────
N_EXAMPLES = 4                       # ↓ reduce to 2 if running short on time
_sample_imgs, _sample_masks = [], []
for imgs, masks in _val_loader_global:
    _sample_imgs.append(imgs)
    _sample_masks.append(masks)
    if sum(len(x) for x in _sample_imgs) >= N_EXAMPLES:
        break
_sample_imgs  = torch.cat(_sample_imgs,  dim=0)[:N_EXAMPLES]   # (N, 3, H, W)
_sample_masks = torch.cat(_sample_masks, dim=0)[:N_EXAMPLES]   # (N, H, W)

# Compute features once for all conditions
_imgs_dev  = _sample_imgs.to(DEVICE)
_feats_dev = extract_backbone_features(_imgs_dev)               # (N, FEAT_DIM, 1, FEAT_SIZE, FEAT_SIZE)

# Predict with each condition (using last-seed heads from Cell 8)
_preds = {}
for cond in CONDITIONS:
    _preds[cond] = predict_mask(
        heads_last[cond], UPSAMPLERS[cond], _imgs_dev, _feats_dev
    )                                # (N, H, W), values 0–149

# ── Plot ──────────────────────────────────────────────────────────────────────
_ncols = 5                           # Original | GT | Bilinear | 2D AnyUp | AnyUp3D
_nrows = N_EXAMPLES                  # ↓ one row per example; depends on N_EXAMPLES
_fig, _axes = plt.subplots(
    _nrows, _ncols,
    figsize=(4 * _ncols, 4 * _nrows),   # ↓ scale with N_EXAMPLES
    squeeze=False,
)
_col_titles = ["Original", "GT", "Bilinear", "2D AnyUp", "AnyUp3D (ours)"]

for col, title in enumerate(_col_titles):
    _axes[0][col].set_title(title, fontsize=14, fontweight='bold', pad=8)

for row in range(_nrows):
    # Original image
    _img_vis = denorm(_sample_imgs[row]).permute(1, 2, 0).numpy()
    _axes[row][0].imshow(_img_vis)

    # GT mask — annotation values 1–150; shift to 0–149 for palette
    _gt_np = _sample_masks[row].numpy().astype(np.int64)
    _axes[row][1].imshow(mask_to_rgb(np.clip(_gt_np, 0, NUM_CLASSES)))

    # Predictions: values 0–149 → add 1 to align with palette (1–150)
    for col_idx, cond in enumerate(CONDITIONS, start=2):
        _pred_np = _preds[cond][row] + 1           # shift to palette space
        _axes[row][col_idx].imshow(mask_to_rgb(_pred_np))

    for ax in _axes[row]:
        ax.axis('off')

_fig.tight_layout(pad=1.0)

_fig_path = os.path.join(OUTPUT_DIR, "exp1_qualitative.png")
_fig.savefig(_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved → {_fig_path}")